# MAC-Fairness: Local Development with Ollama

This notebook demonstrates running multi-agent conversations using Ollama for local development.

## Prerequisites

1. **Ollama**: https://ollama.ai
2. **Model**: `ollama pull llama3.2:1b-instruct-q4_K_M`

```bash
# Setup
uv venv && source .venv/bin/activate
uv pip install -e . ipykernel ipywidgets

# Optional (schemas are for documentation purposes)
cd schema/2025-12-10 && npm install && npm run build && cd ../..
```

## Execution Model

- **Cross-conversation parallelism**: Multiple conversations run concurrently via `asyncio.gather`
- **Dependency-based ordering**: Agents speak based on `speak_after_within_round` config
- **Shared HTTP session**: Connection reuse via `aiohttp.ClientSession`


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Environment variables (must be set before any CUDA imports)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MAC_FAIRNESS_DEBUG_FLAG"] = "1"

## 0. Environment Check


In [ ]:
import subprocess
import sys
import os
from pathlib import Path

# Check Ollama
try:
    result = subprocess.run(
        ["ollama", "list"], capture_output=True, text=True, timeout=5
    )
    print("Ollama: available")
    if "llama3.2:1b" in result.stdout:
        print("Model: llama3.2:1b found")
    else:
        print("Model: missing - run 'ollama pull llama3.2:1b-instruct-q4_K_M'")
except FileNotFoundError:
    print("Ollama: not installed")

# Find project root and set up path
project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / "pyproject.toml").exists():
        break
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
os.chdir(project_root)

In [ ]:
import shutil

cleanup_paths = [
    project_root / "bookkeeping" / "dev_ollama_index.jsonl",
    project_root / "bookkeeping" / "config_snapshot" / "dev_ollama",
    project_root / "experiment" / "dev_ollama",
]

for path in cleanup_paths:
    if path.exists():
        if path.is_file():
            path.unlink()
            print(f"Removed file: {path.relative_to(project_root)}")
        else:
            shutil.rmtree(path)
            print(f"Removed directory: {path.relative_to(project_root)}")
    else:
        print(f"Not found: {path.relative_to(project_root)}")

## 1. Load Configuration


In [ ]:
import json
import yaml

config_path = (
    project_root
    / "config"
    / "dev_ollama"
    / "llama32_1b_3agent_as-human-demographics_vanilla_v2025-12-10_scratch.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

print(f"Experiment: {config['experiment_metadata']['experiment_name']}")
print(f"Schema version: {config['experiment_metadata']['schema_version']}")
print(f"Questions: {config['experiment_metadata']['questions_file']}")

print(f"\nConversation: {config['conversation_config']['routing_strategy']} routing, {config['conversation_config']['max_rounds']} rounds")

ir = config["identity_reveal_config"]
print(f"Identity reveal: persona={ir['reveal_persona']}, demographics={ir['reveal_demographics']}, presence_mode={ir['reveal_presence_mode']}")

pt = config.get("prompt_template_config", {}).get("for_participant", {})
print(f"Prompt template: choice_format={pt.get('choice_display_format', 'bullet')}, json_order={pt.get('json_field_order', 'answer_first')}")

print(f"\nAgents ({len(config['agent_definitions'])}):")
for agent in config["agent_definitions"]:
    print(
        f"  {agent['agent_id']}: persona={agent.get('persona')}, demographics={agent.get('demographics')}, if_as_human={agent.get('if_as_human')}"
    )

## 2. Preview Questions


In [ ]:
questions_file = project_root / config["experiment_metadata"]["questions_file"]
with open(questions_file) as f:
    questions = [json.loads(line) for line in f if line.strip()]

print(f"Total questions: {len(questions)}\n")

q = questions[0]
print(f"Question: {q['question']}")
print(f"Context: {q['context']}")
for c in q["choices"]:
    marker = "->" if c["id"] == q["correct_answer_id"] else "  "
    print(f"{marker} {c['id']}: {c['text']}")

## 2.5 Preview Prompts (Detailed Trajectory)

Inspect the actual prompts that will be sent to agents across rounds for a single question.

NOTE: when showing structured response of an agent, the framework internally convert the response to A/B/C indexed choices.


In [ ]:
from src.prompt.participant import ParticipantPromptBuilder
from src.utils.config_manager import ConfigManager
from src.utils.conversation_orchestrator import ConversationOrchestrator

# Run single question experiment
orchestrator = ConversationOrchestrator(str(config_path))
await orchestrator.run_experiment(question_range=(0, 1))  # just 1 question

In [ ]:
# Load the transcript and display detailed trajectory with ACTUAL prompts
transcript_dir = (
    project_root
    / "experiment"
    / "dev_ollama"
    / config["experiment_metadata"]["experiment_name"]
    / "transcript"
)
latest_transcript = sorted(transcript_dir.glob("*.json"))[-1]

with open(latest_transcript) as f:
    t = json.load(f)

# Question info
print(f"Question: {t['experiment_metadata']['question_id']}")

# Show each round with actual prompts and responses
for rd in t["conversation_rounds"]:
    print(f"\n{'=' * 80}")
    print(f"ROUND {rd['round_id']}")
    print("=" * 80)

    for msg in rd["messages"]:
        agent_id = msg["agent_id"]
        metadata = msg.get("message_metadata", {})

        print(f"\n--- {agent_id} PROMPT ---")
        # Use actual prompt from metadata (now stored in transcript)
        actual_prompt = metadata.get("prompt", "[prompt not stored in this transcript]")
        print(actual_prompt)

        print(f"\n--- {agent_id} RESPONSE ---")
        resp = msg["structured_response"]
        print(f"Answer: {resp.get('opinion')}")
        print(f"Rationale: {resp.get('rationale')}")

print(f"\n{'=' * 80}")
print("FINAL ANSWERS")
print("=" * 80)
for agent_id, answer in t["conversation_summary"]["final_answers"].items():
    print(f"  {agent_id}: {answer}")

## 3. Run Experiment

Use `question_range=(start, end)` for a subset. Omit for all questions.


In [ ]:
os.environ["MAC_FAIRNESS_DEBUG_FLAG"] = "0"  # production

In [ ]:
from src.utils.conversation_orchestrator import ConversationOrchestrator

orchestrator = ConversationOrchestrator(str(config_path))
await orchestrator.run_experiment(question_range=(0, 10))  # first 10 questions

## 4. Job Summary


In [ ]:
exp_root = (
    project_root
    / "experiment"
    / "dev_ollama"
    / config["experiment_metadata"]["experiment_name"]
)

job_summary_dir = exp_root / "job_summary"

latest_summary = sorted(job_summary_dir.glob("*.json"))[-1]
with open(latest_summary) as f:
    job_summary = json.load(f)

proc = job_summary.get("processing_statistics", {})
perf = job_summary.get("throughput_performance", {})

print(
    f"Questions: {proc.get('questions_succeeded', 0)}/{proc.get('questions_attempted', 0)} succeeded"
)
print(f"Throughput: {perf.get('questions_per_second', 0):.3f} q/s")
print(f"Tokens/sec: {perf.get('tokens_per_second', 0):.1f}")